# Data Preprocessing

**Notebook 02 — Phase 1**

## Executive Summary

Clean the dataset, define column roles, perform a stratified train/test split, and persist interim artifacts for downstream notebooks.

## Objectives

- Drop identifiers and constant columns
- Define categorical vs numerical features
- Perform stratified 80/20 split
- Prevent data leakage through split-before-fit discipline

## Expected Outputs

- `data/interim/train.csv` and `test.csv`

## Required Inputs

- Run from the **project root** with the virtual environment activated (`make install`).
- Prerequisites: Notebook 01 (optional); raw CSV available
- Reproducibility: `RANDOM_STATE = 42` where applicable.


## 1. Setup

**Objective:** Load configuration and libraries.

In [1]:
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "data" / "raw").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pandas as pd
from sklearn.model_selection import train_test_split

from notebooks._shared.config import INTERIM_DIR, RANDOM_STATE, TARGET, TEST_SIZE
from notebooks._shared.preprocessing import clean_dataset, get_feature_columns, load_raw_dataset

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data Cleaning

**Objective:** Remove non-predictive columns.

**Methodology:** Drop employee ID and constant fields.

**Why:** Identifiers cause memorization; constants provide zero information.

In [2]:
df_raw = load_raw_dataset()
df_clean = clean_dataset(df_raw)
print(f"Shape before: {df_raw.shape} -> after: {df_clean.shape}")

Shape before: (1470, 35) -> after: (1470, 31)


## 3. Column Selection

**Objective:** Define feature roles before splitting.

**Why:** Explicit column lists prevent target leakage and document modeling choices.

In [3]:
categorical_cols, numerical_cols = get_feature_columns(df_clean, include_engineered=False)
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical ({len(numerical_cols)}): {len(numerical_cols)} columns")

Categorical (7): ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
Numerical (23): 23 columns


## 4. Train/Test Split

**Objective:** Hold out 20% for unbiased evaluation.

**Methodology:** Stratified split on `Attrition` preserves class ratio.

**Why:** Prevents leakage — encoding and scaling will fit on train only in Notebook 04.

In [4]:
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)

train_df = X_train.copy()
train_df[TARGET] = y_train.values
test_df = X_test.copy()
test_df[TARGET] = y_test.values

train_path = INTERIM_DIR / "train.csv"
test_path = INTERIM_DIR / "test.csv"
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Train: {train_df.shape} | Test: {test_df.shape}")
print(f"Train attrition rate: {(y_train == 'Yes').mean():.1%}")
print(f"Test attrition rate: {(y_test == 'Yes').mean():.1%}")
print(f"Saved: {train_path.name}, {test_path.name}")

Train: (1176, 31) | Test: (294, 31)
Train attrition rate: 16.2%
Test attrition rate: 16.0%
Saved: train.csv, test.csv


### Observations

Stratified split maintains ~16% positive class in both partitions.

**Business Interpretation:** Representative test set ensures evaluation reflects real HR prevalence.

**Conclusion:** Interim splits saved for feature engineering and modeling.

## Future Connection

Notebook 03 engineers eight domain-informed HR features on each split independently and saves `train_features.csv` / `test_features.csv`.
